# VideoSeek - Module 2: Query-Adaptive Perception & Dynamic FPS Focus
Notebook này dùng để kiểm thử trên **Google Colab** cho **Module 2: Query-Adaptive Perception & Dynamic FPS Focus** của dự án `VideoSeek`:
1. **Cell 1:** Clone repository `videoseek` và cài đặt các thư viện cần thiết (`decord`, `litellm`, `pillow`, `pyyaml`, `yt-dlp`, `openai-whisper`, `opencv-python-headless`, `pytest`).
2. **Cell 2:** Thiết lập biến môi trường kết nối LLM/VLM API (`OPENAI_API_KEY`, `OPENAI_API_BASE`, `OPENAI_API_VERSION`).
3. **Cell 3:** Chạy bộ **Unit Tests** tự động kiểm tra `QueryAnalyzer`, `Dynamic FPS Focus` (`fps=1, 4, 8`), và `Hybrid Anomaly Overview Sampling`.
4. **Cell 4:** Tạo **Synthetic Video** kiểm thử bằng OpenCV (video dài 120s/1200s có chèn dòng chữ màu đỏ `"BI MAT"` xuất hiện chớp nhoáng trong 3 giây) hoặc tải video thực tế.
5. **Cell 5:** Khởi chạy `VideoSeekAgent` với câu hỏi kiểm thử sự kiện phi cốt truyện, in chi tiết `QueryProfile`, lịch sử `Thought -> Action -> Observation`, và kiểm chứng Agent gọi `focus` với `fps=8`.

## Cell 1: Clone Repository & Cài đặt Thư viện trên Google Colab

In [11]:
import os
import sys

# 1. Clone repo nếu đang chạy trên Google Colab và chưa có thư mục videoseek
if not os.path.exists("videoseek"):
    !git clone https://github.com/nhatsonle/videoseek.git

# Di chuyển vào thư mục gốc của repo videoseek
if os.path.exists("videoseek") and os.path.isdir("videoseek") and os.path.exists("videoseek/config"):
    os.chdir("videoseek")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Current working directory:", os.getcwd())

# 2. Cài đặt đầy đủ các thư viện yêu cầu (bao gồm cả mcp cho litellm tool calling)
!pip install -q decord litellm mcp pillow pyyaml yt-dlp openai-whisper opencv-python-headless pytest

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.47 KiB | 754.00 KiB/s, done.
From https://github.com/nhatsonle/videoseek
 * branch            main       -> FETCH_HEAD
   477c139..4ea05bb  main       -> origin/main
Updating 477c139..4ea05bb
Fast-forward
 test.ipynb | 79 +++++++++++++++++++++++++++++++++++++++-----------------------
 1 file changed, 50 insertions(+), 29 deletions(-)
Current working directory: /content/videoseek


## Cell 2: Cấu hình Biến Môi trường API (`OPENAI_API_KEY`, `OPENAI_API_BASE`, `OPENAI_API_VERSION`)

In [16]:
import os
from config import general_config, prompts_config

# =====================================================================
# CẤU HÌNH API & MODEL TẠI ĐÂY:
# Cách 1: Điền trực tiếp API_KEY (sk-...) bên dưới
# Cách 2: Thêm 'OPENAI_API_KEY' vào mục Secrets (biểu tượng chìa khóa 🔑 bên trái Google Colab)
# =====================================================================
API_KEY = "sk-IDLnMUwgPABlgfNlakxNSw"  # <-- Thay bằng key 'sk-...' của bạn
API_BASE = "https://hub.smilesoftware.org/v1"
API_VERSION = ""
MODEL_NAME = "gpt-6.1-terra"  # Có thể đổi thành "openai/gpt-6.1-terra", "gpt-5.6-terra", "gpt-4o", v.v.

def _resolve_config_value(secret_name: str, direct_value: str, placeholder: str = "YOUR_OPENAI_API_KEY") -> str:
    # 1. Nếu người dùng đã điền trực tiếp giá trị hợp lệ (khác placeholder) -> ưu tiên dùng luôn
    if direct_value and direct_value != placeholder:
        return direct_value
    # 2. Thử đọc từ Google Colab Secrets
    try:
        from google.colab import userdata
        secret_val = userdata.get(secret_name)
        if secret_val and secret_val != placeholder:
            return secret_val
    except Exception:
        pass
    # 3. Thử đọc từ biến môi trường (bỏ qua nếu đang bị dính giá trị placeholder cũ)
    env_val = os.getenv(secret_name)
    if env_val and env_val != placeholder:
        return env_val
    return direct_value

os.environ["OPENAI_API_KEY"] = _resolve_config_value("OPENAI_API_KEY", API_KEY)
os.environ["OPENAI_API_BASE"] = _resolve_config_value("OPENAI_API_BASE", API_BASE)
os.environ["OPENAI_API_VERSION"] = _resolve_config_value("OPENAI_API_VERSION", API_VERSION)

# Chuẩn bị runtime config cho VideoSeekAgent
runtime_config = {**general_config, **prompts_config}
runtime_config["api_key"] = os.environ["OPENAI_API_KEY"]
runtime_config["api_base"] = os.environ["OPENAI_API_BASE"]
runtime_config["api_version"] = os.environ["OPENAI_API_VERSION"] or None
runtime_config["model_name"] = _resolve_config_value("VIDEOSEEK_MODEL", MODEL_NAME)

masked_key = runtime_config["api_key"][:6] + "..." + runtime_config["api_key"][-4:] if len(runtime_config["api_key"]) > 10 else runtime_config["api_key"]
print("Configured Model:", runtime_config["model_name"])
print("Configured API Base:", runtime_config["api_base"])
print("Configured API Key:", masked_key)
if runtime_config["api_key"] == "YOUR_OPENAI_API_KEY" or not runtime_config["api_key"].startswith("sk-"):
    print("⚠️ CẢNH BÁO: Bạn chưa thay API_KEY hợp lệ (bắt đầu bằng 'sk-') ở Cell 2!")
print("Adaptive FPS Enabled:", runtime_config.get("adaptive_fps_enabled"))
print("Anomaly Overview Enabled:", runtime_config.get("anomaly_overview_enabled"))
print("Max Focus Frames Cap:", runtime_config.get("max_focus_frames_cap"))

Configured Model: gpt-6.1-terra
Configured API Base: https://hub.smilesoftware.org/v1
Configured API Key: sk-IDL...xNSw
Adaptive FPS Enabled: True
Anomaly Overview Enabled: True
Max Focus Frames Cap: 24


## Cell 3: Chạy Bộ Unit Tests Tự Động Kiểm Tra Module 2 (`unittest` / `pytest`)
Kiểm tra toàn diện:
1. `QueryAnalyzer`: Phân loại câu hỏi qua 1 LLM JSON Router duy nhất (`reasoning_effort="low"`, `return_json=True`) và cơ chế fallback an toàn khi lỗi API/JSON.
2. `focus_tool` & `execute_focus`: Schema bắt buộc có `fps`, tính toán số frame theo `duration_window * fps`, giới hạn bởi `max_focus_frames_cap=24`, xử lý an toàn `start_time >= end_time`, và format timestamp `:.2f` khi `fps >= 4`.
3. `execute_overview` & `_sample_hybrid_anomaly_frames`: Lấy mẫu lai 75% Uniform + 25% Anomaly Peak (tổng số frame chia hết cho 8) và bắt trúng sự kiện màu đỏ chớp nhoáng.

In [17]:
import os
import sys
import importlib
import importlib.util
import unittest

# Đảm bảo đang đứng ở thư mục gốc của videoseek và kéo bản cập nhật mới nhất từ GitHub
if not os.path.exists("tests/test_module2.py"):
    if os.path.exists("videoseek") and os.path.isdir("videoseek"):
        os.chdir("videoseek")
    !git pull origin main

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Reload lại các module của videoseek nếu đã được import từ bản cũ trước khi git pull
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

# Nạp trực tiếp file tests/test_module2.py (tránh xung đột tên gói 'tests' có sẵn trên Google Colab)
test_file_path = os.path.abspath("tests/test_module2.py")
spec = importlib.util.spec_from_file_location("test_module2", test_file_path)
test_module2 = importlib.util.module_from_spec(spec)
sys.modules["test_module2"] = test_module2
spec.loader.exec_module(test_module2)

TestModule2AdaptivePerception = test_module2.TestModule2AdaptivePerception

# Chạy toàn bộ 8 kịch bản Unit Tests từ tests/test_module2.py
suite = unittest.TestLoader().loadTestsFromTestCase(TestModule2AdaptivePerception)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit tests failed!"
print(f"\n[PASS] Toàn bộ {result.testsRun}/{result.testsRun} Unit Tests của Module 2 đã vượt qua thành công!")

test_1_query_analyzer_4_types_and_safe_coercion (test_module2.TestModule2AdaptivePerception.test_1_query_analyzer_4_types_and_safe_coercion)
Test 1: LLM JSON Router for all 4 query_types + safe type coercion for strings. ... ok
test_2_query_analyzer_fallback_on_errors (test_module2.TestModule2AdaptivePerception.test_2_query_analyzer_fallback_on_errors)
Test 2: LLM JSON Router safe fallback on None, malformed JSON, or missing fields. ... ok
test_3_focus_schema_and_timestamp_formatting (test_module2.TestModule2AdaptivePerception.test_3_focus_schema_and_timestamp_formatting)
Test 3: Focus tool strict schema and .2f vs .1f timestamp formatting. ... ok
test_4_focus_edge_cases (test_module2.TestModule2AdaptivePerception.test_4_focus_edge_cases)
Test 4: Focus edge cases (start_time >= end_time, > video_duration, max_focus_frames_cap=24). ... ok
test_5_hybrid_anomaly_overview_20min_video_at_420_423s (test_module2.TestModule2AdaptivePerception.test_5_hybrid_anomaly_overview_20min_video_at_420_4

QueryAnalyzer fallback to default profile due to error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


ok
test_7_end_to_end_agent_and_clean_trajectory (test_module2.TestModule2AdaptivePerception.test_7_end_to_end_agent_and_clean_trajectory)
Test 7: End-to-End VideoSeekAgent.run() through overview -> focus(fps=8) -> answer, ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.581s

OK



[PASS] Toàn bộ 7/7 Unit Tests của Module 2 đã vượt qua thành công!


## Cell 4: Tạo Video Kiểm Thử Giả Lập (Synthetic Video bằng OpenCV)
Tạo một video kiểm thử (mặc định `120s`, có thể đổi `SYNTHETIC_DURATION_SEC = 1200` cho video 20 phút) gồm các cảnh chuyển động thông thường, nhưng tại khoảng **`44.0s - 47.0s`** (3 giây) xuất hiện đột ngột dòng chữ màu đỏ nổi bật **`"BI MAT"`** không có trong phụ đề `.srt`.

In [18]:
import cv2
import numpy as np

SYNTHETIC_VIDEO_PATH = "synthetic_anomaly_test.mp4"
SYNTHETIC_SRT_PATH = "synthetic_anomaly_test.srt"
SYNTHETIC_DURATION_SEC = 120  # Có thể chỉnh thành 1200 để test video 20 phút
SYNTHETIC_FPS = 10            # 10 FPS giúp tạo video nhanh trên Colab mà vẫn đủ kiểm thử focus 8 FPS
ANOMALY_START_SEC = 44.0
ANOMALY_END_SEC = 47.0


def create_synthetic_test_video(
    video_path: str = SYNTHETIC_VIDEO_PATH,
    srt_path: str = SYNTHETIC_SRT_PATH,
    duration_sec: int = SYNTHETIC_DURATION_SEC,
    fps: int = SYNTHETIC_FPS,
    anomaly_start: float = ANOMALY_START_SEC,
    anomaly_end: float = ANOMALY_END_SEC,
):
    width, height = 640, 360
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))
    total_frames = int(duration_sec * fps)

    for idx in range(total_frames):
        t = idx / float(fps)
        # Nền xám xanh dịu có hình tròn trắng di chuyển chậm mô phỏng mạch video bình thường
        frame = np.full((height, width, 3), (45, 55, 65), dtype=np.uint8)
        cx = int((t / duration_sec) * (width - 100)) + 50
        cy = height // 2
        cv2.circle(frame, (cx, cy), 30, (200, 200, 200), -1)
        cv2.putText(
            frame,
            f"Time: {t:.1f}s",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (180, 180, 180),
            2,
        )

        # Chèn sự kiện phi cốt truyện: chữ màu đỏ "BI MAT" trong 3 giây (44.0s - 47.0s)
        if anomaly_start <= t <= anomaly_end:
            cv2.rectangle(frame, (80, 100), (560, 260), (0, 0, 40), -1)
            cv2.rectangle(frame, (80, 100), (560, 260), (0, 0, 255), 6)
            cv2.putText(
                frame,
                "BI MAT",
                (155, 205),
                cv2.FONT_HERSHEY_SIMPLEX,
                3.0,
                (0, 0, 255),  # BGR: Đỏ tươi
                8,
            )

        writer.write(frame)

    writer.release()

    # Tạo file phụ đề mô phỏng cốt truyện bình thường (không hề nhắc tới chữ "BI MAT")
    srt_content = (
        "1\n"
        "00:00:00,000 --> 00:00:30,000\n"
        "Phần mở đầu giới thiệu chuyển động cơ bản của vật thể trên màn hình.\n\n"
        "2\n"
        "00:00:30,000 --> 00:01:30,000\n"
        "Vật thể hình tròn tiếp tục di chuyển đều từ trái sang phải.\n\n"
        "3\n"
        "00:01:30,000 --> 00:02:00,000\n"
        "Kết thúc quá trình quan sát chuyển động đều.\n"
    )
    with open(srt_path, "w", encoding="utf-8") as f:
        f.write(srt_content)

    print(f"Đã tạo video kiểm thử: {video_path} ({duration_sec}s @ {fps} FPS)")
    print(f"Sự kiện chữ đỏ 'BI MAT' được chèn tại: {anomaly_start}s - {anomaly_end}s")


create_synthetic_test_video()

Đã tạo video kiểm thử: synthetic_anomaly_test.mp4 (120s @ 10 FPS)
Sự kiện chữ đỏ 'BI MAT' được chèn tại: 44.0s - 47.0s


## Cell 5: Khởi Chạy `VideoSeekAgent` & Kiểm Chứng `QueryProfile` + `focus(fps=8)`
Chạy `VideoSeekAgent` với câu hỏi tìm dòng chữ màu đỏ xuất hiện chớp nhoáng trong video, hiển thị:
1. Kết quả phân tích câu hỏi từ `QueryAnalyzer` (`QueryProfile`).
2. Toàn bộ các bước suy luận `Thought -> Action -> Observation`.
3. Kiểm chứng Agent có sử dụng `focus` với `fps=8` (hoặc `fps >= 4`) và trả lời đúng `"BI MAT"`.

In [20]:
try:
    import mcp
except ImportError:
    !pip install -q mcp
!git pull origin main
import importlib, sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

from config import general_config, prompts_config
from videoseek.agent import VideoSeekAgent

# Đồng bộ lại SYSTEM_PROMPT và các tham số mới nhất vào runtime_config sau khi reload
runtime_config.update(prompts_config)
for k, v in general_config.items():
    runtime_config.setdefault(k, v)

test_question = (
    "Trong video có một dòng chữ in hoa màu đỏ xuất hiện chớp nhoáng trong khoảng 3 giây "
    "(không nằm trong phụ đề). Nội dung chính xác của dòng chữ màu đỏ đó là gì?"
)

agent = VideoSeekAgent(
    config=runtime_config,
    video_path=SYNTHETIC_VIDEO_PATH,
    subtitle_path=SYNTHETIC_SRT_PATH,
    output_dir="./outputs",
    tools=["overview", "skim", "focus"],
    verbose=True,
)

# Chạy Agent trên câu hỏi kiểm thử
trajectory = agent.run(test_question)

print("\n================ KẾT QUẢ KIỂM CHỨNG MODULE 2 ================")
if agent.query_profile is not None:
    print("1. QueryProfile từ LLM JSON Router:")
    for k, v in agent.query_profile.to_dict().items():
        print(f"   - {k}: {v}")

print("\n2. Tóm tắt Trajectory (Thought -> Action -> Observation):")
focus_fps_used = []
for step in trajectory.steps:
    fn_name = step.action.function_name
    params = step.action.parameters
    if fn_name == "focus":
        focus_fps_used.append(params.get("fps"))
    print(f"   [Step {step.step_id}] Action: {fn_name} | Params: {params}")

print("\n3. Kiểm chứng Dynamic FPS Focus:")
print(f"   - Các giá trị FPS đã truyền vào tool `focus`: {focus_fps_used}")
if any(fps == 8 for fps in focus_fps_used if fps is not None):
    print("   -> ĐẠT: Agent đã chủ động gọi `focus` với fps=8 cho sự kiện chữ chớp nhoáng!")
else:
    print("   -> Lưu ý: Kiểm tra lại log trajectory phía trên.")

print(f"\n4. Final Answer: {trajectory.final_answer}")
print(f"   Finish Reason: {trajectory.finish_reason}")

remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 13 (delta 9), reused 13 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 10.41 KiB | 1.30 MiB/s, done.
From https://github.com/nhatsonle/videoseek
 * branch            main       -> FETCH_HEAD
   4ea05bb..efd0530  main       -> origin/main
Updating 4ea05bb..efd0530
Fast-forward
 config/prompts.yaml   |   3 +-
 pyproject.toml        |   1 +
 test.ipynb            | 384 ++++++++++++++++++++++++++++++++++++++++++++------
 tests/test_module2.py | 196 +++++++++++++++++++++++++-
 videoseek/agent.py    | 161 ++++++++++++++++++---
 5 files changed, 676 insertions(+), 69 deletions(-)
--------------------------------
Video ID: synthetic_anomaly_test (120.0s)
--------------------------------
Question:
Trong video có một dòng chữ in hoa màu đỏ xuất hiện chớp nhoáng trong khoảng 3 giây (không nằm trong phụ đề). Nội dung chính xác của dòn

## Cell 6: Tự Upload Video Của Bạn (hoặc Đường Dẫn File / Link YouTube) & Chạy Kiểm Thử
Bạn có thể cung cấp video theo **1 trong 3 cách**:
1. **Upload trực tiếp từ máy tính:** Để `CUSTOM_VIDEO_PATH = ""`, khi chạy cell sẽ tự động bật nút **Choose Files** của Google Colab để bạn chọn file video (`.mp4`, `.avi`, `.mov`, `.mkv`) và file phụ đề `.srt` (nếu có).
2. **Kéo thả file vào thanh Files bên trái Colab (hoặc Google Drive):** Điền đường dẫn vào `CUSTOM_VIDEO_PATH = "/content/ten_video.mp4"`.
3. **Dán trực tiếp Link YouTube:** Điền URL YouTube vào `CUSTOM_VIDEO_PATH = "https://www.youtube.com/watch?v=..."`, hệ thống sẽ tự động tải video về bằng `yt-dlp`.

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from videoseek.agent import VideoSeekAgent

# =====================================================================
# NHẬP VIDEO VÀ CÂU HỎI CỦA BẠN TẠI ĐÂY:
# - Để trống CUSTOM_VIDEO_PATH = "" nếu muốn bấm nút Upload file từ máy tính
# - Hoặc điền đường dẫn file trên Colab (vd: "/content/my_video.mp4") hoặc link YouTube
# =====================================================================
CUSTOM_VIDEO_PATH = ""  # Ví dụ: "/content/video_test.mp4" hoặc "https://www.youtube.com/watch?v=uBIe_NCUDuQ"
CUSTOM_SRT_PATH = ""    # Tuỳ chọn: Đường dẫn file phụ đề .srt (để trống nếu video không có phụ đề)
CUSTOM_QUESTION = "Đồ vật gì được giấu đằng sau bức chân dung Hitler treo trên tường?"

def prepare_custom_video(video_input: str, srt_input: str = ""):
    video_path = video_input.strip()
    srt_path = srt_input.strip() if srt_input else None

    # Trường hợp 1: Để trống -> Mở hộp thoại Upload của Google Colab
    if not video_path:
        try:
            from google.colab import files
            print("📂 Vui lòng chọn file video (.mp4, .mkv, .mov, .avi) và file .srt (nếu có) từ máy tính của bạn:")
            uploaded = files.upload()
            for fname in uploaded.keys():
                abs_path = os.path.abspath(fname)
                if fname.lower().endswith(".srt"):
                    srt_path = abs_path
                elif fname.lower().endswith((".mp4", ".mkv", ".mov", ".avi", ".webm")):
                    video_path = abs_path
            if not video_path and uploaded:
                video_path = os.path.abspath(next(iter(uploaded.keys())))
        except Exception as e:
            raise RuntimeError(
                f"Không thể mở hộp thoại upload tự động ({e}). "
                "Hãy kéo thả video vào mục Files của Colab và điền đường dẫn vào biến CUSTOM_VIDEO_PATH."
            )

    # Trường hợp 2: Là link YouTube -> Tự động tải bằng yt-dlp
    elif video_path.startswith(("http://", "https://")) and ("youtube.com" in video_path or "youtu.be" in video_path):
        dl_dir = Path("./custom_downloads")
        dl_dir.mkdir(parents=True, exist_ok=True)
        output_template = str(dl_dir / "%(id)s.%(ext)s")
        print(f"🌐 Đang tải video từ YouTube: {video_path} ...")
        cmd = [
            "yt-dlp",
            "-f", "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best",
            "--merge-output-format", "mp4",
            "-o", output_template,
            video_path,
        ]
        subprocess.run(cmd, check=True)
        mp4_files = sorted(dl_dir.glob("*.mp4"), key=os.path.getmtime, reverse=True)
        if not mp4_files:
            raise FileNotFoundError("Không tìm thấy file .mp4 sau khi tải bằng yt-dlp.")
        video_path = str(mp4_files[0].resolve())

    if not video_path or not os.path.exists(video_path):
        raise FileNotFoundError(f"Không tìm thấy file video tại: '{video_path}'")

    if srt_path and not os.path.exists(srt_path):
        print(f"⚠️ Không tìm thấy file phụ đề '{srt_path}', hệ thống sẽ chạy chế độ không dùng phụ đề.")
        srt_path = None

    return video_path, srt_path

# 1. Chuẩn bị video và câu hỏi
resolved_video_path, resolved_srt_path = prepare_custom_video(CUSTOM_VIDEO_PATH, CUSTOM_SRT_PATH)
print(f"✅ Video sử dụng: {resolved_video_path}")
print(f"✅ Phụ đề (.srt): {resolved_srt_path or 'Không có (chỉ dùng thị giác)'}")

# 2. Khởi tạo và chạy VideoSeekAgent
custom_agent = VideoSeekAgent(
    config=runtime_config,
    video_path=resolved_video_path,
    subtitle_path=resolved_srt_path,
    output_dir="./outputs",
    tools=["overview", "skim", "focus"],
    verbose=True,
)

custom_trajectory = custom_agent.run(CUSTOM_QUESTION)

# 3. In tổng kết và lưu kết quả ra file JSON
print("\n================ KẾT QUẢ CHẠY TRÊN VIDEO CỦA BẠN ================")
if custom_agent.query_profile is not None:
    print("1. QueryProfile (LLM JSON Router):", json.dumps(custom_agent.query_profile.to_dict(), ensure_ascii=False))

print(f"2. Tổng số bước thực hiện: {len(custom_trajectory.steps)}")
for step in custom_trajectory.steps:
    print(f"   - [Step {step.step_id}] Tool: {step.action.function_name} | Params: {step.action.parameters}")

print(f"\n3. Câu trả lời cuối cùng (Final Answer):\n{custom_trajectory.final_answer}")
print(f"4. Finish Reason: {custom_trajectory.finish_reason}")

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/custom_trajectory.json", "w", encoding="utf-8") as f:
    json.dump(custom_trajectory.to_dict(), f, ensure_ascii=False, indent=2)
print("💾 Đã lưu chi tiết quỹ đạo vào: ./outputs/custom_trajectory.json")

## Cell 7: A/B Benchmark So Sánh Hiệu Năng Trước & Sau Khi Nâng Cấp (Cùng 1 Video & 1 Câu Hỏi)
Cell này tự động thực thi **2 lượt chạy đối đầu trực tiếp** trên cùng một Video và cùng một Câu hỏi:
1. **Lượt 1 - Baseline (Trước nâng cấp - VideoSeek gốc):** Chạy với `adaptive_fps_enabled=False`, `anomaly_overview_enabled=False`.
2. **Lượt 2 - Upgraded (Sau nâng cấp - AdaSeek):** Chạy với `adaptive_fps_enabled=True`, `anomaly_overview_enabled=True` (kích hoạt LLM JSON Router, Hybrid Overview, Dynamic FPS Focus).

Sau khi chạy xong, hệ thống sẽ in ra **Bảng So Sánh Hiệu Năng Đối Đầu (Head-to-Head Comparison)**: đo lường số turns, thời gian xử lý (giây), chuỗi công cụ đã gọi, và độ chính xác của đáp án.

In [ ]:
import time
import copy
import json
import os
import importlib
import sys
from pathlib import Path

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

from config import general_config, prompts_config
from videoseek.agent import VideoSeekAgent

runtime_config.update(prompts_config)
for k, v in general_config.items():
    runtime_config.setdefault(k, v)

# =====================================================================
# CẤU HÌNH BÀI KIỂM THỬ A/B BENCHMARK:
# - Mặc định dùng video kiểm thử 120s có chữ đỏ 3s (hoặc đổi thành video của bạn)
# - Nhập câu hỏi muốn thử nghiệm đối đầu
# =====================================================================
BENCH_VIDEO_PATH = SYNTHETIC_VIDEO_PATH  # Hoặc đường dẫn video tùy ý (vd: "/content/my_video.mp4")
BENCH_SRT_PATH = SYNTHETIC_SRT_PATH      # Hoặc đường dẫn file .srt (để trống "" để tự động tìm file .srt cùng tên)
BENCH_QUESTION = (
    "Trong video có một dòng chữ in hoa màu đỏ xuất hiện chớp nhoáng trong khoảng 3 giây "
    "(không nằm trong phụ đề). Nội dung chính xác của dòng chữ màu đỏ đó là gì?"
)

# Tự động tìm file phụ đề .srt cùng tên với BENCH_VIDEO_PATH nếu có
if not BENCH_SRT_PATH or not os.path.exists(str(BENCH_SRT_PATH)):
    candidate_srt = str(Path(BENCH_VIDEO_PATH).with_suffix(".srt"))
    if os.path.exists(candidate_srt):
        BENCH_SRT_PATH = candidate_srt
        print(f"📄 Tự động tìm thấy file phụ đề cùng tên: {BENCH_SRT_PATH}")
    else:
        BENCH_SRT_PATH = None

print("=======================================================================")
print("🏁 BẮT ĐẦU A/B BENCHMARK: SO SÁNH TRỰC DIỆN TRƯỚC VÀ SAU NÂNG CẤP")
print(f"📹 Video: {BENCH_VIDEO_PATH}")
print(f"📄 Phụ đề (.srt): {BENCH_SRT_PATH or 'Không có'}")
print(f"❓ Question: {BENCH_QUESTION}")
print("=======================================================================\n")

# ---------------------------------------------------------------------
# 1. CHẠY LƯỢT 1: BASELINE (TRƯỚC NÂNG CẤP - VIDEOSEEK GỐC)
# ---------------------------------------------------------------------
print("⏳ [1/2] ĐANG CHẠY BASELINE (Cơ chế VideoSeek gốc: Tắt Router, Lấy mẫu đều thô, Focus cố định 1 FPS)...")
cfg_baseline = copy.deepcopy(runtime_config)
cfg_baseline["adaptive_fps_enabled"] = False
cfg_baseline["anomaly_overview_enabled"] = False

agent_baseline = VideoSeekAgent(
    config=cfg_baseline,
    video_path=BENCH_VIDEO_PATH,
    subtitle_path=BENCH_SRT_PATH if BENCH_SRT_PATH else None,
    output_dir="./outputs/baseline",
    tools=["overview", "skim", "focus"],
    verbose=True,
)

t0_base = time.time()
traj_base = agent_baseline.run(BENCH_QUESTION)
dur_base = time.time() - t0_base
steps_base = len(traj_base.steps)
tools_base = " -> ".join([s.action.function_name for s in traj_base.steps]) if traj_base.steps else "None"
ans_base = traj_base.final_answer.strip() if traj_base.final_answer else "None"
print(f"   └── Xong Baseline: {steps_base} turns | Thời gian: {dur_base:.1f}s | Finish: {traj_base.finish_reason}")

# ---------------------------------------------------------------------
# 2. CHẠY LƯỢT 2: UPGRADED (SAU NÂNG CẤP - ADASEEK)
# ---------------------------------------------------------------------
print("\n🚀 [2/2] ĐANG CHẠY UPGRADED (AdaSeek: Bật LLM JSON Router, Hybrid Overview, Dynamic FPS Focus)...")
cfg_upgraded = copy.deepcopy(runtime_config)
cfg_upgraded["adaptive_fps_enabled"] = True
cfg_upgraded["anomaly_overview_enabled"] = True

agent_upgraded = VideoSeekAgent(
    config=cfg_upgraded,
    video_path=BENCH_VIDEO_PATH,
    subtitle_path=BENCH_SRT_PATH if BENCH_SRT_PATH else None,
    output_dir="./outputs/upgraded",
    tools=["overview", "skim", "focus"],
    verbose=True,
)

t0_upg = time.time()
traj_upg = agent_upgraded.run(BENCH_QUESTION)
dur_upg = time.time() - t0_upg
steps_upg = len(traj_upg.steps)
tools_upg = " -> ".join([s.action.function_name for s in traj_upg.steps]) if traj_upg.steps else "None"
ans_upg = traj_upg.final_answer.strip() if traj_upg.final_answer else "None"
qprofile_upg = agent_upgraded.query_profile.query_type if agent_upgraded.query_profile else "N/A"
rec_fps_upg = agent_upgraded.query_profile.recommended_fps if agent_upgraded.query_profile else 1.0
print(f"   └── Xong Upgraded: {steps_upg} turns | Thời gian: {dur_upg:.1f}s | Finish: {traj_upg.finish_reason}")

# ---------------------------------------------------------------------
# 3. BẢNG SO SÁNH HIỆU NĂNG ĐỐI ĐẦU CHI TIẾT (HEAD-TO-HEAD REPORT)
# ---------------------------------------------------------------------
print("\n" + "=" * 75)
print("📊 BẢNG SO SÁNH HIỆU NĂNG ĐỐI ĐẦU (HEAD-TO-HEAD BENCHMARK)")
print("=" * 75)
header = f"{'Tiêu Chí Đánh Giá':<26} | {'BASELINE (Trước nâng cấp)':<22} | {'UPGRADED (Sau nâng cấp)':<22}"
print(header)
print("-" * 75)
print(f"{'Cơ chế Query Router':<26} | {'Không có (Mù câu hỏi)':<22} | {f'{qprofile_upg} (FPS={rec_fps_upg:g})':<22}")
print(f"{'Cơ chế Lấy Mẫu Overview':<26} | {'100% Đều (Mắt lưới thô)':<22} | {'Hybrid (75% Đều + 25% Đỉnh)':<22}")
print(f"{'Hỗ trợ Dynamic FPS':<26} | {'Không (Cố định 1 FPS)':<22} | {'Có (1, 2, 4, 8 FPS)':<22}")
print(f"{'Số Turns Suy Luận':<26} | {f'{steps_base} turns':<22} | {f'{steps_upg} turns':<22}")
print(f"{'Thời Gian Thực Thi':<26} | {f'{dur_base:.1f} giây':<22} | {f'{dur_upg:.1f} giây':<22}")
print(f"{'Trạng Thái Dừng':<26} | {str(traj_base.finish_reason):<22} | {str(traj_upg.finish_reason):<22}")
print(f"{'Chuỗi Tool Đã Gọi':<26} | {tools_base:<22} | {tools_upg:<22}")
print("-" * 75)
short_base = (ans_base[:20] + '...') if len(ans_base) > 20 else ans_base
short_upg = (ans_upg[:20] + '...') if len(ans_upg) > 20 else ans_upg
print(f"{'Đáp Án Cuối Cùng':<26} | {short_base:<22} | {short_upg:<22}")
print("=" * 75)

# Đánh giá định tính
print("\n💡 NHẬN XÉT ĐÁNH GIÁ CHUYÊN MÔN:")
if steps_upg < steps_base:
    pct_turns = ((steps_base - steps_upg) / steps_base) * 100
    print(f"• Tiết kiệm số lượt gọi (Turns): Giảm {pct_turns:.1f}% số turn so với Baseline!")
if "BI MAT" in ans_upg.upper() and "BI MAT" not in ans_base.upper():
    print("• Đột phá tri giác: Upgraded BẮT CHÍNH XÁC dòng chữ đỏ 3s mà Baseline bị lọt lưới 100%!")

# Lưu báo cáo vào file JSON
report_data = {
    "video": BENCH_VIDEO_PATH,
    "question": BENCH_QUESTION,
    "baseline": {
        "turns": steps_base,
        "duration_sec": round(dur_base, 2),
        "finish_reason": traj_base.finish_reason,
        "tools_called": tools_base,
        "prediction": ans_base,
    },
    "upgraded": {
        "query_type": qprofile_upg,
        "recommended_fps": rec_fps_upg,
        "turns": steps_upg,
        "duration_sec": round(dur_upg, 2),
        "finish_reason": traj_upg.finish_reason,
        "tools_called": tools_upg,
        "prediction": ans_upg,
    }
}
os.makedirs("./outputs", exist_ok=True)
with open("./outputs/ab_comparison_report.json", "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)
print("💾 Đã lưu toàn bộ số liệu so sánh vào: ./outputs/ab_comparison_report.json")